In [1]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
import os
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report
from torch.utils.data import DataLoader

In [2]:
OUTPUT_MODEL_NAME = "synth_lora_model_gemma"
CHECKPOINT_DIR = "checkpoints_gemma"
TENSORBOARD_RUN_NAME = "synthetic_data_experiment_gemma"

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

# Make constant variables.
MODEL_NAME = "google/embeddinggemma-300m"

In [4]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

In [5]:
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}

dataset = dataset.map(lambda x: {"label": label_map[x["needs"]]})


label_feature = ClassLabel(names=["met", "unmet"])
dataset = dataset.cast_column("label", label_feature)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [6]:
# Tokenize texts.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["report"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize)
 # Set PyTorch format. 
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [7]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")


Train distribution:
label
0    0.502162
1    0.497838
Name: proportion, dtype: float64
Validation distribution:
label
0    0.50173
1    0.49827
Name: proportion, dtype: float64
Test distribution:
label
0    0.502591
1    0.497409
Name: proportion, dtype: float64


In [8]:
# Confirm device for use. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# Load the base model.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Define LoRA config.
lora_config = LoraConfig(
    r=8, # LoRA attention dimension (rank)
    lora_alpha=16, # alpha for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # specific named modules to be replaced (query, key, value, attention output projections)
    lora_dropout=0.05, # dropout probability for LoRA layers
    bias="none", # bias type
    task_type="SEQ_CLS" # what type of task (sequence classification)
)

# Attach LoRA to model. 
model = get_peft_model(model, lora_config)
model.to(device)
model.print_trainable_parameters()

Device: cuda


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Gemma3TextForSequenceClassification LOAD REPORT from: google/embeddinggemma-300m
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 984,576 || all params: 303,849,216 || trainable%: 0.3240


In [9]:
# Set training arguments.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    learning_rate=2e-5,
    num_train_epochs=50, # higher so early stopping can trigger
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    run_name=TENSORBOARD_RUN_NAME,
    fp16=True
)

# Add early stopping.
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

# Add metrics function.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Create trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks = [early_stopping],
    compute_metrics=compute_metrics
)

In [10]:
# Train the model.
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.278100,0.295015,0.925606,0.945455,0.902778,0.923623
2,0.209743,0.304190,0.935986,0.911475,0.965278,0.937605
3,0.180579,0.458978,0.942907,0.907348,0.986111,0.945092


TrainOutput(global_step=3471, training_loss=0.24457893822180096, metrics={'train_runtime': 6075.0426, 'train_samples_per_second': 38.074, 'train_steps_per_second': 9.523, 'total_flos': 2185434586939392.0, 'train_loss': 0.24457893822180096, 'epoch': 3.0})

In [11]:
# Save the fine-tuned model.
model.save_pretrained(OUTPUT_MODEL_NAME)
tokenizer.save_pretrained(OUTPUT_MODEL_NAME)

('synth_lora_model_gemma/tokenizer_config.json',
 'synth_lora_model_gemma/tokenizer.json')

In [12]:
# Move model to eval mode.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

loader = DataLoader(test_dataset, batch_size=32)

preds = []
labels = []

with torch.no_grad():
    for batch in loader:
        # Move all tensors to the same device as the model
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        batch_labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=1)

        preds.extend(predictions.cpu().numpy())   # move to CPU for metrics
        labels.extend(batch_labels.cpu().numpy())

preds = np.array(preds)
labels = np.array(labels)

# Confusion Matrix
cm = confusion_matrix(labels, preds)
print("--------------- Confusion Matrix ---------------")
print(cm)

# Classification Report
report = classification_report(labels, preds)
print("--------------- Classification Report ---------------")
print(report)

--------------- Confusion Matrix ---------------
[[284   7]
 [ 21 267]]
--------------- Classification Report ---------------
              precision    recall  f1-score   support

           0       0.93      0.98      0.95       291
           1       0.97      0.93      0.95       288

    accuracy                           0.95       579
   macro avg       0.95      0.95      0.95       579
weighted avg       0.95      0.95      0.95       579

